# Task 015: lensless_admm — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Lensless imaging reconstruction using ADMM (Alternating Direction Method of Multipliers)

**Paper**: LenslessPiCam: A Hardware and Software Platform for Lensless Computational Imaging with a Raspberry Pi
**Repository**: https://github.com/LCAV/LenslessPiCam

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 8.74 dB
- **SSIM**: 0.1039
- **Evaluation**: Visual reconstruction quality (lensless ADMM)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import fft
from scipy.fftpack import next_fast_len
from lensless.utils.io import load_data
from lensless.utils.plot import plot_image


# ==============================================================================
# Helper Class: RealFFTConvolve2D
# ==============================================================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
def load_and_preprocess_data(psf_path, data_path, downsample=4):
    """
    Load PSF and measurement data, preprocess them for reconstruction.
    
    Parameters
    ----------
    psf_path : str
        Path to the PSF image file.
    data_path : str
        Path to the measurement/raw data image file.
    downsample : int
        Downsampling factor for loading data.
    
    Returns
    -------
    dict
        Dictionary containing 'psf' and 'data' numpy arrays.
    """
    print(f"Loading data from {data_path}...")
    print(f"Loading PSF from {psf_path}...")
    
    psf, data = load_data(
        psf_fp=psf_path,
        data_fp=data_path,
        background_fp=None,
        dtype="float32",
        downsample=downsample,
        bayer=False,
        plot=False,
        flip=False,
        normalize=True
    )
    
    print(f"Data shape: {data.shape}")
    print(f"PSF shape: {psf.shape}")
    
    return {"psf": psf, "data": data}

## 4. Class: `RealFFTConvolve2D`

2D convolution in Fourier domain, with same real-valued kernel.

Methods: `__init__`, `_crop`, `_pad`, `set_psf`, `convolve`, `deconvolve`

In [ ]:
class RealFFTConvolve2D:
    """
    2D convolution in Fourier domain, with same real-valued kernel.
    """
    def __init__(self, psf, dtype=np.float32, pad=True, norm="ortho"):
        self.dtype = dtype
        self.pad = pad
        self.norm = norm
        self.set_psf(psf)

    def _crop(self, x):
        return x[..., self._start_idx[0] : self._end_idx[0], self._start_idx[1] : self._end_idx[1], :]

    def _pad(self, v):
        if len(v.shape) == 5:
            batch_size = v.shape[0]
            shape = [batch_size] + self._padded_shape
        elif len(v.shape) == 4:
            shape = self._padded_shape
        else:
            raise ValueError("Expected 4D or 5D tensor")

        vpad = np.zeros(shape).astype(v.dtype)
        vpad[..., self._start_idx[0] : self._end_idx[0], self._start_idx[1] : self._end_idx[1], :] = v
        return vpad

    def set_psf(self, psf):
        self._psf = psf.astype(self.dtype)
        self._psf_shape = np.array(self._psf.shape)

        self._padded_shape = 2 * self._psf_shape[-3:-1] - 1
        self._padded_shape = np.array([next_fast_len(int(i)) for i in self._padded_shape])
        self._padded_shape = list(np.r_[self._psf_shape[-4], self._padded_shape, self._psf.shape[-1]].astype(int))
        
        self._start_idx = ((np.array(self._padded_shape[-3:-1]) - self._psf_shape[-3:-1]) // 2).astype(int)
        self._end_idx = (self._start_idx + self._psf_shape[-3:-1]).astype(int)

        self._H = fft.rfft2(self._pad(self._psf), axes=(-3, -2), norm=self.norm)
        self._Hadj = np.conj(self._H)
        self._padded_data = np.zeros(self._padded_shape).astype(self.dtype)

    def convolve(self, x):
        if self.pad:
            self._padded_data = self._pad(x)
        else:
            self._padded_data[:] = x

        conv_output = fft.rfft2(self._padded_data, axes=(-3, -2)) * self._H
        conv_output = fft.ifftshift(
            fft.irfft2(conv_output, axes=(-3, -2), s=self._padded_shape[-3:-1]),
            axes=(-3, -2),
        )
        
        if self.pad:
            conv_output = self._crop(conv_output)
            
        return conv_output

    def deconvolve(self, y):
        if self.pad:
            self._padded_data = self._pad(y)
        else:
            self._padded_data[:] = y

        deconv_output = fft.rfft2(self._padded_data, axes=(-3, -2)) * self._Hadj
        deconv_output = fft.ifftshift(
            fft.irfft2(deconv_output, axes=(-3, -2), s=self._padded_shape[-3:-1]),
            axes=(-3, -2),
        )

        if self.pad:
            deconv_output = self._crop(deconv_output)
            
        return deconv_output

## 5. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(image_est, psf):
    """
    Apply the forward imaging model: convolve the estimated image with the PSF.
    
    Parameters
    ----------
    image_est : np.ndarray
        Estimated image, shape [D, H, W, C] or [1, D, H, W, C].
    psf : np.ndarray
        Point spread function, shape [D, H, W, C].
    
    Returns
    -------
    np.ndarray
        Simulated measurement (convolution result).
    """
    convolver = RealFFTConvolve2D(psf, dtype=psf.dtype, pad=True)
    return convolver.convolve(image_est)

## 6. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

Functions: `run_inversion`

In [ ]:
def run_inversion(data_dict, n_iter=50, mu1=1e-6, mu2=1e-5, mu3=4e-5, tau=0.0001):
    """
    Run ADMM-based inversion to reconstruct the image from measurements.
    
    Parameters
    ----------
    data_dict : dict
        Dictionary containing 'psf' and 'data'.
    n_iter : int
        Number of ADMM iterations.
    mu1, mu2, mu3 : float
        ADMM penalty parameters.
    tau : float
        TV regularization weight.
    
    Returns
    -------
    np.ndarray
        Reconstructed image.
    """
    psf = data_dict["psf"]
    measurement = data_dict["data"]
    dtype = np.float32
    
    # Initialize convolver with pad=False for internal ADMM operations
    convolver = RealFFTConvolve2D(psf, dtype=dtype, pad=False)
    padded_shape = convolver._padded_shape
    psf_shape = convolver._psf_shape
    
    # Helper functions for finite differences (TV prior)
    def finite_diff(x):
        return np.stack(
            (np.roll(x, 1, axis=-3) - x, np.roll(x, 1, axis=-2) - x),
            axis=len(x.shape),
        )
    
    def finite_diff_adj(x):
        diff1 = np.roll(x[..., 0], -1, axis=-3) - x[..., 0]
        diff2 = np.roll(x[..., 1], -1, axis=-2) - x[..., 1]
        return diff1 + diff2
    
    def finite_diff_gram(shape, dt):
        gram = np.zeros(shape, dtype=dt)
        if shape[0] == 1:
            gram[0, 0, 0] = 4
            gram[0, 0, 1] = gram[0, 0, -1] = gram[0, 1, 0] = gram[0, -1, 0] = -1
        else:
            gram[0, 0, 0] = 6
            gram[0, 0, 1] = gram[0, 0, -1] = gram[0, 1, 0] = gram[0, -1, 0] = gram[1, 0, 0] = gram[-1, 0, 0] = -1
        return fft.rfft2(gram, axes=(-3, -2))
    
    # Precompute TV Gram matrix
    PsiTPsi = finite_diff_gram(padded_shape, dtype)
    
    # Initialize variables in padded space
    image_est = np.zeros([1] + padded_shape, dtype=dtype)
    
    # Prepare data: pad measurement to match padded field
    data_padded = convolver._pad(measurement)
    
    # ADMM variables
    X = np.zeros_like(image_est)
    U = np.zeros_like(finite_diff(image_est))
    W = np.zeros_like(X)
    
    xi = np.zeros_like(image_est)
    eta = np.zeros_like(U)
    rho = np.zeros_like(X)
    
    # Precompute division matrices
    H = convolver._H
    Hadj = convolver._Hadj
    
    denom = mu1 * (np.abs(Hadj * H)) + mu2 * np.abs(PsiTPsi) + mu3
    R_divmat = 1.0 / denom.astype(np.complex64)
    
    X_divmat = 1.0 / (convolver._pad(np.ones(list(psf_shape.astype(int)), dtype=dtype)) + mu1)
    
    print(f"Starting ADMM for {n_iter} iterations...")
    start_time = time.time()
    
    for i in range(n_iter):
        if i % 10 == 0:
            print(f"  Iteration {i}/{n_iter}")
        
        # 1. U update (TV Soft Thresholding)
        Psi_out = finite_diff(image_est)
        U = np.sign(Psi_out + eta / mu2) * np.maximum(0, np.abs(Psi_out + eta / mu2) - tau / mu2)
        
        # 2. X update
        forward_out = convolver.convolve(image_est)
        X = X_divmat * (xi + mu1 * forward_out + data_padded)
        
        # 3. W update (Non-negativity)
        W = np.maximum(rho / mu3 + image_est, 0)
        
        # 4. Image update (Frequency domain)
        rk = (
            (mu3 * W - rho)
            + finite_diff_adj(mu2 * U - eta)
            + convolver.deconvolve(mu1 * X - xi)
        )
        
        freq_result = R_divmat * fft.rfft2(rk, axes=(-3, -2))
        image_est = fft.irfft2(freq_result, axes=(-3, -2), s=convolver._padded_shape[-3:-1])
        
        # 5. Lagrangian updates
        forward_out = convolver.convolve(image_est)
        Psi_out = finite_diff(image_est)
        
        xi += mu1 * (forward_out - X)
        eta += mu2 * (Psi_out - U)
        rho += mu3 * (image_est - W)
    
    end_time = time.time()
    print(f"Reconstruction finished in {end_time - start_time:.2f}s")
    
    # Crop result
    result = convolver._crop(image_est)
    
    # Remove batch dimension if present
    if result.shape[0] == 1:
        result = result[0]
    
    return result

## 7. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(reconstruction, output_path="result.png"):
    """
    Evaluate and save the reconstruction result.
    
    Parameters
    ----------
    reconstruction : np.ndarray
        Reconstructed image.
    output_path : str
        Path to save the output image.
    """
    print(f"Reconstruction shape: {reconstruction.shape}")
    print(f"Reconstruction min: {reconstruction.min():.4f}, max: {reconstruction.max():.4f}")
    
    print(f"Saving result to {output_path}...")
    ax = plot_image(reconstruction, gamma=None)
    if hasattr(ax, "__len__"):
        ax = ax[0, 0]
    ax.set_title("ADMM Reconstruction")
    plt.savefig(output_path)
    plt.close()
    
    npy_path = output_path.replace(".png", ".npy")
    np.save(npy_path, reconstruction)
    print(f"Saved numpy array to {npy_path}")
    
    # Compute some basic metrics
    mean_val = np.mean(reconstruction)
    std_val = np.std(reconstruction)
    print(f"Reconstruction statistics - Mean: {mean_val:.4f}, Std: {std_val:.4f}")

## 8. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
psf_file = "data/psf/tape_rgb.png"
data_file = "data/raw_data/thumbs_up_rgb.png"

if not os.path.exists(psf_file):
    print(f"Error: {psf_file} not found.")
    exit(1)

if not os.path.exists(data_file):
    print(f"Error: {data_file} not found.")
    exit(1)

### Step 1: Load and preprocess data

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Load and preprocess data
data_dict = load_and_preprocess_data(psf_file, data_file, downsample=4)

### Step 2: Run inversion

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 2: Run inversion
reconstruction = run_inversion(data_dict, n_iter=5)

### Step 3: Evaluate results

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 3: Evaluate results
evaluate_results(reconstruction, "admm_result_refactored.png")

# Demonstrate forward operator
print("\nDemonstrating forward operator...")
if len(reconstruction.shape) == 3:
    recon_4d = reconstruction[np.newaxis, ...]
else:
    recon_4d = reconstruction

simulated_measurement = forward_operator(recon_4d, data_dict["psf"])
print(f"Forward operator output shape: {simulated_measurement.shape}")

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 9. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **lensless_admm**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=8.74 dB, SSIM=0.1039)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: LenslessPiCam: A Hardware and Software Platform for Lensless Computational Imaging with a Raspberry Pi
- Repository: https://github.com/LCAV/LenslessPiCam
